# Linux Networking Fundamentals (Educational Notebook)
This notebook introduces core Linux networking concepts: interfaces, IP addressing, routing, DNS resolution, and the everyday command-line tools used to inspect and troubleshoot a network connection.

## 1. Network Interfaces

A network interface is the kernel's handle for a physical or virtual network device. Every interface has a name, a link-layer (MAC) address, and one or more IP addresses assigned to it.

Common interface naming patterns:
- `lo` - the loopback interface (`127.0.0.1` / `::1`), always present, used for local-only traffic.
- `eth0`, `eth1` - traditional kernel-assigned names for wired Ethernet devices.
- `enp0s3`, `ens33`, `enX...` - "predictable network interface names" used by modern `systemd`/`udev`, based on bus/slot position so names don't shuffle across reboots.
- `wlan0`, `wlp2s0` - wireless interfaces.
- `docker0`, `veth...`, `br-...` - virtual interfaces created by container runtimes and bridges.
- `tun0`, `tap0` - virtual point-to-point interfaces used by VPNs.

```bash
ip link show          # list interfaces and their administrative/operational state
ip -br link            # brief, one-line-per-interface summary
ip link set eth0 up     # bring an interface up
ip link set eth0 down   # take an interface down
```


## 2. IP Addressing

Every interface needs at least one IP address to participate in a network.

- **IPv4** addresses are 32 bits, written as four dotted decimal octets (e.g. `192.168.1.10`).
- **IPv6** addresses are 128 bits, written as eight groups of hex digits separated by colons (e.g. `2001:db8::1`), with `::` used once to compress a run of zero groups.
- **CIDR notation** (`192.168.1.0/24`) combines an address with a prefix length that says how many leading bits belong to the network portion. `/24` means the first 24 bits are the network, leaving 8 bits (256 addresses) for hosts.
- **Private (RFC 1918) ranges** used inside LANs and never routed on the public internet: `10.0.0.0/8`, `172.16.0.0/12`, `192.168.0.0/16`.
- **Loopback**: `127.0.0.0/8` (IPv4), `::1` (IPv6).
- **Link-local**: `169.254.0.0/16` (IPv4 auto-assigned when DHCP fails), `fe80::/10` (IPv6, always present on an interface).

```bash
ip addr show            # list all addresses on all interfaces
ip -4 addr show eth0     # IPv4 addresses on one interface
ip addr add 192.168.1.10/24 dev eth0   # assign an address
ip addr del 192.168.1.10/24 dev eth0   # remove an address
```


## 3. Routing and the Default Gateway

The kernel decides where to send each outgoing packet by consulting its **routing table**. Routes are matched by longest-prefix (most specific network) first; if nothing more specific matches, the **default route** (`0.0.0.0/0`) is used, which points at the **default gateway** - usually your router.

```bash
ip route show             # print the routing table
ip route show default      # print just the default route
ip route get 8.8.8.8        # ask the kernel which route it would use for a destination
ip route add 10.0.0.0/24 via 192.168.1.1 dev eth0   # add a static route
ip route del 10.0.0.0/24                             # remove a route
```

A typical routing table entry looks like:

```
default via 192.168.1.1 dev eth0 proto dhcp metric 100
192.168.1.0/24 dev eth0 proto kernel scope link src 192.168.1.10
```

The first line says "anything not matched elsewhere goes to `192.168.1.1`". The second line says "the `192.168.1.0/24` network is directly reachable on `eth0`, no gateway needed". When multiple routes could match, the one with the lowest `metric` wins.


## 4. DNS Resolution

DNS turns hostnames into IP addresses. On a Linux host, name resolution normally checks several sources in order, controlled by `/etc/nsswitch.conf`:

- `/etc/hosts` - a plain-text file mapping hostnames to IPs, checked first by default. Useful for local overrides and testing.
- `/etc/resolv.conf` - lists the DNS server(s) (`nameserver` lines) and search domains to use if `/etc/hosts` has no entry. On many modern distros this file is managed automatically by `systemd-resolved` or `NetworkManager`.
- `/etc/nsswitch.conf` - the line `hosts: files dns` defines the actual lookup order (`files` = `/etc/hosts` first, then `dns`).

```bash
cat /etc/hosts               # local hostname -> IP overrides
cat /etc/resolv.conf          # configured DNS servers
resolvectl status              # DNS status on systemd-resolved systems
getent hosts example.com        # resolve a name using the full NSS chain
dig example.com                  # query DNS directly, bypassing /etc/hosts
nslookup example.com              # older DNS query tool
```


## 5. Inspecting the Network with `ip` and `ss`

The `iproute2` package (`ip`, `ss`, `tc`) is the modern, actively maintained toolkit for Linux networking and is preinstalled on essentially every current distribution.

```bash
ip addr                 # addresses on every interface (alias: ip a)
ip link                  # interfaces and their link state (alias: ip l)
ip route                  # routing table (alias: ip r)
ip neigh                   # ARP/neighbor table (alias: ip n)

ss -tulpn                   # listening TCP (t) and UDP (u) sockets, with process (p) and numeric ports (n)
ss -tan                       # all TCP connections, numeric
ss -s                          # summary statistics
```


## 6. Legacy Tools: `ifconfig`, `netstat`, `route`

These come from the older `net-tools` package. They are deprecated on most modern distributions (not installed by default) but are still common in older documentation, scripts, and exam material, so it's worth recognizing them.

| Legacy command | Modern replacement |
|---|---|
| `ifconfig` | `ip addr`, `ip link` |
| `route` | `ip route` |
| `netstat -tulpn` | `ss -tulpn` |
| `arp -a` | `ip neigh` |

```bash
ifconfig                # (legacy) list interfaces and addresses
netstat -tulpn            # (legacy) listening sockets
route -n                   # (legacy) routing table, numeric
```


## 7. Connectivity Testing: `ping`, `traceroute`, `curl`, `wget`

```bash
ping -c 4 example.com        # send 4 ICMP echo requests, report round-trip time
traceroute example.com        # show the router hops a packet takes to reach a destination
tracepath example.com          # traceroute-like tool that doesn't need root privileges
mtr example.com                 # combines ping + traceroute into a live, continuously updating view

curl -I https://example.com       # fetch just the HTTP headers, useful for checking a service is up
curl -v https://example.com        # verbose output, shows the TLS handshake and full request/response
wget https://example.com/file.tar   # download a file
```

`ping` tests basic reachability at the IP layer. `traceroute`/`mtr` show *where* a connection is failing or slow along the path. `curl`/`wget` test at the application layer (HTTP), confirming a service is not just reachable but actually responding correctly.


## 8. Firewall Basics: `iptables` and `nftables`

Linux filters packets using rules organized into **chains**, evaluated in order. The traditional tool is `iptables` (built on the `netfilter` kernel framework); `nftables` is its modern successor and is now the default on most distributions, though it can still be driven through an `iptables`-compatible command on many systems.

Key built-in chains:
- **INPUT** - packets destined for this host.
- **OUTPUT** - packets originating from this host.
- **FORWARD** - packets being routed through this host (relevant when the host acts as a router).

```bash
iptables -L -n -v              # list current rules, numeric, with packet/byte counters
iptables -A INPUT -p tcp --dport 22 -j ACCEPT   # allow inbound SSH
iptables -A INPUT -j DROP                        # drop everything else not already accepted

nft list ruleset                 # list current nftables rules
nft add rule inet filter input tcp dport 22 accept   # equivalent allow-SSH rule in nftables syntax
```

Rules are evaluated top to bottom within a chain; the first matching rule decides the packet's fate (unless it uses a non-terminating target). A common mistake is adding an `ACCEPT` rule *after* a `DROP` rule that already matches - it will never be reached.


## 9. Common Ports and Services

Every TCP/UDP connection is identified by a source and destination **port** in addition to the IP address. Well-known ports (0-1023) are reserved for standard services:

```
20/21   FTP (data / control)
22      SSH
23      Telnet
25      SMTP
53      DNS
67/68   DHCP (server / client)
80      HTTP
110     POP3
123     NTP
143     IMAP
443     HTTPS
3306    MySQL
5432    PostgreSQL
```

```bash
ss -tulpn | grep :22        # confirm what's actually listening on a given port
cat /etc/services             # the full local port-to-service name mapping
```


## 10. Network Troubleshooting Workflow

When a connection isn't working, work outward from the local host toward the remote service:

1. **Interface up?** `ip link show` - is the interface `UP` and does it show `LOWER_UP`?
2. **Address assigned?** `ip addr show` - does the interface have a sane IP address?
3. **Local network reachable?** `ping` the default gateway.
4. **Routing correct?** `ip route get <destination>` - what route would the kernel actually use?
5. **Internet reachable?** `ping 8.8.8.8` (tests IP-layer connectivity without involving DNS).
6. **DNS working?** `dig example.com` or `getent hosts example.com` (tests name resolution specifically, isolated from steps 3-5).
7. **Port reachable?** `curl -v` or a tool like `nc -zv host port` (tests that the specific service is listening and responding).
8. **Firewall interfering?** `iptables -L -n -v` or `nft list ruleset` on both ends of the connection.

Working through these in order isolates *which layer* is broken - link, IP, DNS, or application - instead of guessing.


## Hands-on

Try these on your own Linux system (a VM is safest if you're going to touch firewall rules):

```bash
ip addr show
ip route show
cat /etc/resolv.conf
ss -tulpn
ping -c 3 8.8.8.8
dig example.com
traceroute example.com
iptables -L -n -v
```

For each command, note what it told you and which of the OSI/TCP-IP layers (link, network, transport, application) the information belongs to.


## Review Questions

1. What is the difference between `ip link` and `ip addr`?
2. What does the `/24` in `192.168.1.0/24` mean?
3. Why does a host need a default gateway, and what happens if no route matches a destination and there is no default route?
4. In what order does a typical Linux host check `/etc/hosts` versus DNS servers, and what file controls that order?
5. Name two modern replacements for the legacy `ifconfig` and `netstat` commands.
6. What is the practical difference between what `ping` tests and what `curl` tests?
7. In `iptables`, why does rule order inside a chain matter?
8. Which chain (`INPUT`, `OUTPUT`, or `FORWARD`) applies to a packet being routed through a Linux box acting as a router, rather than to or from the box itself?
9. If `ping 8.8.8.8` works but `ping example.com` fails, which part of the troubleshooting workflow points you at the problem?
10. What port does HTTPS use by default, and how would you confirm nothing else is already listening on it before starting a web server?


# Cheat Sheet

```
Interfaces:      ip link show | ip -br link
Addresses:       ip addr show | ip addr add <cidr> dev <if>
Routing:         ip route show | ip route get <dest>
Neighbors/ARP:   ip neigh

Sockets:         ss -tulpn | ss -tan | ss -s
Legacy equivs:   ifconfig -> ip addr/link, route -> ip route, netstat -> ss

Connectivity:    ping -c4 <host> | traceroute <host> | mtr <host>
HTTP checks:     curl -I <url> | curl -v <url> | wget <url>

DNS files:       /etc/hosts, /etc/resolv.conf, /etc/nsswitch.conf
DNS tools:       dig <host> | nslookup <host> | getent hosts <host> | resolvectl status

Firewall:        iptables -L -n -v | nft list ruleset
Common ports:    22 SSH, 53 DNS, 80 HTTP, 443 HTTPS, 3306 MySQL, 5432 PostgreSQL
```
